---

### **模型评估与选择教案实验素材**

---

#### **一、数据划分方法**

---


##### **1. 留出法（Hold-out）**
**原理**：将数据集划分为互斥的训练集和测试集，常用比例为7:3或8:2。  
**适用场景**：数据量较大时，简单高效。  
**代码示例**：  
```python
from sklearn.model_selection import train_test_split

# 分类数据示例（X为特征，y为标签）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3,   # 测试集比例（默认0.25）
    random_state=42, # 随机种子，确保结果可复现
    stratify=y       # 分层抽样，保持类别比例（仅用于分类任务）
)

# 回归数据示例（无需stratify）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42
)
```


---

##### **2. k折交叉验证（k-Fold Cross Validation）**
**原理**：将数据集划分为k个子集，每次用k-1个子集训练，剩余1个子集测试，重复k次。  
**适用场景**：数据量较小时，充分利用数据。  
**代码示例**：  
```python
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# 初始化模型
model = LogisticRegression()

# 分类任务：5折交叉验证
kfold = KFold(
    n_splits=5,      # 折数（默认5）
    shuffle=True,     # 是否打乱数据顺序
    random_state=42   # 随机种子
)
scores = cross_val_score(
    model, X, y, 
    cv=kfold,         # 指定交叉验证策略
    scoring='accuracy' # 评估指标（分类：'accuracy', 'f1'；回归：'neg_mean_squared_error'）
)
print("交叉验证平均准确率:", scores.mean())

# 回归任务（使用决策树）
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor()
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    model, X, y, 
    cv=kfold, 
    scoring='neg_mean_squared_error'  # 负均方误差（Scikit-learn约定）
)
print("交叉验证平均MSE:", -scores.mean())  # 转换为正数
```


---

##### **3. 自助法（Bootstrap）**
**原理**：通过有放回抽样生成多个训练集，未被抽中的样本作为测试集。  
**适用场景**：小数据集，估计统计量偏差。  
**代码示例**：  
```python
from sklearn.utils import resample
from sklearn.metrics import accuracy_score


# 生成自助样本
n_iterations = 100  # 重复次数
accuracies = []

for _ in range(n_iterations):
    # 有放回抽样
    X_train = resample(X, replace=True, n_samples=len(X), random_state=42)
    y_train = resample(y, replace=True, n_samples=len(y), random_state=42)
    
    # 未被抽中的样本作为测试集
    mask = np.isin(X.index, X_train.index, invert=True)
    X_test, y_test = X[mask], y[mask]
    
    # 训练模型并评估
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if len(y_test) > 0:  # 避免测试集为空
        accuracies.append(accuracy_score(y_test, y_pred))

print("自助法平均准确率:", np.mean(accuracies))
```

---



#### **二、评估指标**

---

##### **1. 回归任务评估指标**
**常用指标**：  
- **均方误差（MSE）**：预测值与真实值差的平方均值，越小越好。  
- **均方根误差（RMSE）**：MSE的平方根，量纲与原始数据一致。  
- **平均绝对误差（MAE）**：预测值与真实值差的绝对值均值，鲁棒性强。  
- **R²分数**：模型解释的方差比例，越接近1越好。  

**代码示例**：  
```python
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_true = [3, 5, 2.5, 7]
y_pred = [2.5, 5, 4, 8]

# 计算指标
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"MSE: {mse:.2f}, RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.2f}")
```

---

##### **2. 分类任务评估指标**
**常用指标**：  
- **准确率（Accuracy）**：分类正确的样本比例。  
- **精确率（Precision）**：预测为正类的样本中实际为正类的比例。  
- **召回率（Recall）**：实际为正类的样本中被正确预测的比例。  
- **F1分数**：精确率和召回率的调和平均数。  
- **AUC-ROC曲线**：反映模型对正负样本的区分能力。  

**代码示例**：  
```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_true = [1, 0, 1, 1, 0]
y_pred = [1, 0, 0, 1, 1]
y_prob = [0.9, 0.2, 0.6, 0.8, 0.7]  # 预测概率（用于AUC）

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

print(f"准确率: {accuracy:.2f}, 精确率: {precision:.2f}, 召回率: {recall:.2f}, F1: {f1:.2f}, AUC: {auc:.2f}")
```





# 模型评估与选择教案

## 一、数据划分方法
### 1. 留出法（Hold-Out）
**原理**：将数据集划分为训练集和测试集，比例通常为 7:3 或 8:2  
**关键参数**：
• `test_size`：测试集比例（0.2表示20%）
• `random_state`：随机种子，保证可复现性

```python
from sklearn.model_selection import train_test_split

# 案例：Iris数据集划分
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=42
)
```

### 2. K折交叉验证（K-Fold CV）
**原理**：将数据分为K个子集，每次用K-1个子集训练，剩余1个验证  
**关键参数**：
• `n_splits`：折数（常用5或10）
• `shuffle`：是否打乱数据顺序

```python
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_index, val_index in kf.split(X):
    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]
```

### 3. 自助法（Bootstrap）
**原理**：有放回采样生成训练集，未选中的样本作为测试集  
**特点**：适合小数据集，但会改变数据分布

```python
from sklearn.utils import resample

# 生成自助样本
X_train = resample(X, n_samples=100, replace=True, random_state=42)
# 获取未选中的样本作为测试集
X_test = np.array([x for x in X if x not in X_train])
```

---

## 二、评估指标
### 1. 回归任务
• **MSE（均方误差）**：`sklearn.metrics.mean_squared_error`
• **MAE（平均绝对误差）**：`sklearn.metrics.mean_absolute_error`
• **R²（决定系数）**：`sklearn.metrics.r2_score`

```python
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)  # 越小越好
print(f"MSE: {mse:.2f}")
```

### 2. 分类任务
• **准确率**：`accuracy_score`
• **精确率/召回率/F1**：`precision_score`, `recall_score`, `f1_score`
• **AUC-ROC**：`roc_auc_score`

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))  # 输出详细分类指标

# AUC计算（需预测概率值）
y_proba = model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_proba)
```

---

```python
# 混淆矩阵可视化示例
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
```
